# Texas Crime Geospatial Analysis
## Interactive Notebook Walkthrough

This notebook demonstrates the full analysis pipeline step by step.
All outputs use the **synthetic dataset** so no data downloads are required.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

print('Setup complete.')

## 1. Load & Preprocess Data

In [ ]:
from src.python.data.loader import generate_synthetic_dataset
from src.python.data.preprocessor import (
    clean_incident_gdf, add_severity, add_crime_category
)

# Generate synthetic Texas crime data
raw_gdf = generate_synthetic_dataset(n_incidents=5000, seed=42)
gdf = clean_incident_gdf(raw_gdf)
gdf = add_severity(gdf)
gdf = add_crime_category(gdf)

print(f'Incidents: {len(gdf):,}')
print(f'Columns:   {list(gdf.columns)}')
gdf.head(3)

In [ ]:
# Crime type distribution
gdf['offense_type'].value_counts().head(10)

## 2. Static Visualisations

In [ ]:
from src.python.visualization.heatmap import plot_crime_type_bar
fig = plot_crime_type_bar(gdf, save=False)
import matplotlib.pyplot as plt
plt.show()

In [ ]:
from src.python.analysis.statistical_analysis import temporal_trend
from src.python.visualization.heatmap import plot_temporal_trend

df = gdf.drop(columns=['geometry'])
df['offense_count'] = 1
trend = temporal_trend(df)
print(f'Trend: {trend.trend_direction}  (p={trend.mk_p_value:.4f})')

fig = plot_temporal_trend(trend.df, save=False)
plt.show()

## 3. Kernel Density Estimation (Hotspot Detection)

In [ ]:
from src.python.analysis.hotspot_detection import compute_kde
from src.python.visualization.heatmap import plot_kde_surface

kde = compute_kde(gdf, grid_size=150, hotspot_pct=90)
print(f'KDE bandwidth: {kde.bandwidth:.4f}')
print(f'Hotspot cells: {kde.hotspot_mask.sum()} / {kde.hotspot_mask.size}')

fig = plot_kde_surface(kde, save=False)
plt.show()

## 4. DBSCAN Spatial Clustering

In [ ]:
from src.python.analysis.spatial_clustering import run_dbscan
from src.python.visualization.heatmap import plot_cluster_scatter

db = run_dbscan(gdf, eps_deg=0.08, min_pts=8)
print(f'Clusters : {db.n_clusters}')
print(f'Noise    : {db.n_noise}')
print(f'C++ used : {db.used_cpp}')

fig = plot_cluster_scatter(db.gdf, save=False)
plt.show()

## 5. K-Means Clustering & Elbow Analysis

In [ ]:
from src.python.analysis.spatial_clustering import run_kmeans, elbow_analysis

elbow_df = elbow_analysis(gdf, k_range=range(2, 12))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(elbow_df['k'], elbow_df['inertia'], 'o-', color='steelblue')
ax.set_xlabel('k'); ax.set_ylabel('Inertia')
ax.set_title('K-Means Elbow Analysis')
plt.tight_layout(); plt.show()

km = run_kmeans(gdf, k=8)
print(f'K-Means inertia (k=8): {km.inertia:.1f}')

## 6. Ripley's K Function

In [ ]:
from src.python.analysis.spatial_clustering import ripleys_k
from src.python.visualization.heatmap import plot_ripleys_l

rk = ripleys_k(gdf, r_values=np.linspace(0.01, 1.5, 40))
fig = plot_ripleys_l(rk, save=False)
plt.show()

peak_r = rk.loc[rk['L_minus_r'].idxmax(), 'r']
print(f'Peak clustering scale: {peak_r:.3f}° (~{peak_r*111:.0f} km)')

## 7. Predictive Risk Modeling

In [ ]:
from src.python.analysis.spatial_clustering import compute_hexbins
from src.python.analysis.predictive_model import (
    build_feature_matrix, train_random_forest
)
from src.python.visualization.heatmap import plot_feature_importance

# Aggregate to hexagonal grid
hex_result = compute_hexbins(gdf, cell_size=0.4)
hex_gdf = hex_result.hex_gdf
hex_gdf['offense_count'] = hex_gdf['count']

X, y, names = build_feature_matrix(hex_gdf, 'offense_count',
                                    include_spatial_lag=False)
result = train_random_forest(X, y, names, n_estimators=50, cv_folds=3)

print(f'CV RMSE : {result.cv_rmse_mean:.2f} ± {result.cv_rmse_std:.2f}')
print(f'CV R²   : {result.cv_r2_mean:.4f}')

fig = plot_feature_importance(result.feature_importances, save=False)
plt.show()

## 8. Interactive Maps

The interactive maps are saved as HTML files – click the link to open them.

In [ ]:
from src.python.visualization.map_generator import kde_heatmap_map, cluster_map
from IPython.display import IFrame

kde_heatmap_map(gdf, out_name='nb_kde_heatmap')
cluster_map(db.gdf, db.cluster_gdf, out_name='nb_cluster_map')

print('Maps saved to outputs/maps/')
IFrame('../outputs/maps/nb_kde_heatmap.html', width=900, height=500)

## 9. Launch Dashboard

Uncomment and run to launch the interactive Dash dashboard.

In [ ]:
# from src.python.visualization.dashboard import create_app
# app = create_app(gdf)
# app.run(debug=False, port=8050)
# # Then open http://127.0.0.1:8050